# Denoising Autoencoder using MNIST

In [ ]:
# ============================================================
# Denoising Autoencoder using MNIST
# ============================================================

import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
import numpy as np

# -----------------------------
# Load Dataset
# -----------------------------
(X_train, _), (X_test, _) = keras.datasets.mnist.load_data()

print("Training Images :", X_train.shape)
print("Testing Images  :", X_test.shape)

# -----------------------------
# Normalize Images
# -----------------------------
X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

# -----------------------------
# Reshape Images
# -----------------------------
X_train = X_train.reshape(-1, 28, 28, 1)
X_test = X_test.reshape(-1, 28, 28, 1)

# -----------------------------
# Add Gaussian Noise
# -----------------------------
noise_factor = 0.3

X_train_noisy = X_train + noise_factor * np.random.normal(
    loc=0.0,
    scale=1.0,
    size=X_train.shape
)

X_test_noisy = X_test + noise_factor * np.random.normal(
    loc=0.0,
    scale=1.0,
    size=X_test.shape
)

X_train_noisy = np.clip(X_train_noisy, 0., 1.)
X_test_noisy = np.clip(X_test_noisy, 0., 1.)

# -----------------------------
# Display Noisy Images
# -----------------------------
plt.figure(figsize=(10,6))

for i in range(8):

    plt.subplot(2,4,i+1)
    plt.imshow(
        X_train_noisy[i].reshape(28,28),
        cmap="gray"
    )
    plt.axis("off")

plt.suptitle("Noisy Images")
plt.tight_layout()
plt.show()

# -----------------------------
# Build Autoencoder
# -----------------------------
input_img = keras.layers.Input(shape=(28,28,1))

# Encoder
x = keras.layers.Conv2D(
    32,
    (3,3),
    activation="relu",
    padding="same"
)(input_img)

x = keras.layers.MaxPooling2D(
    (2,2),
    padding="same"
)(x)

x = keras.layers.Conv2D(
    16,
    (3,3),
    activation="relu",
    padding="same"
)(x)

encoded = keras.layers.MaxPooling2D(
    (2,2),
    padding="same"
)(x)

# Decoder
x = keras.layers.Conv2D(
    16,
    (3,3),
    activation="relu",
    padding="same"
)(encoded)

x = keras.layers.UpSampling2D((2,2))(x)

x = keras.layers.Conv2D(
    32,
    (3,3),
    activation="relu",
    padding="same"
)(x)

x = keras.layers.UpSampling2D((2,2))(x)

decoded = keras.layers.Conv2D(
    1,
    (3,3),
    activation="sigmoid",
    padding="same"
)(x)

autoencoder = keras.Model(
    input_img,
    decoded
)

# -----------------------------
# Model Summary
# -----------------------------
autoencoder.summary()

# -----------------------------
# Compile Model
# -----------------------------
autoencoder.compile(
    optimizer="adam",
    loss="binary_crossentropy"
)

# -----------------------------
# Train Model
# -----------------------------
history = autoencoder.fit(
    X_train_noisy,
    X_train,
    epochs=15,
    batch_size=128,
    validation_data=(
        X_test_noisy,
        X_test
    ),
    verbose=1
)

# -----------------------------
# Predict Denoised Images
# -----------------------------
decoded_images = autoencoder.predict(X_test_noisy)

# -----------------------------
# Plot Loss
# -----------------------------
plt.figure(figsize=(8,5))

plt.plot(
    history.history["loss"],
    label="Training Loss"
)

plt.plot(
    history.history["val_loss"],
    label="Validation Loss"
)

plt.title("Binary Crossentropy Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.show()

# -----------------------------
# Display Results
# -----------------------------
n = 8

plt.figure(figsize=(15,6))

for i in range(n):

    # Original
    plt.subplot(3,n,i+1)
    plt.imshow(
        X_test[i].reshape(28,28),
        cmap="gray"
    )
    plt.title("Original")
    plt.axis("off")

    # Noisy
    plt.subplot(3,n,i+n+1)
    plt.imshow(
        X_test_noisy[i].reshape(28,28),
        cmap="gray"
    )
    plt.title("Noisy")
    plt.axis("off")

    # Denoised
    plt.subplot(3,n,i+2*n+1)
    plt.imshow(
        decoded_images[i].reshape(28,28),
        cmap="gray"
    )
    plt.title("Recovered")
    plt.axis("off")

plt.tight_layout()
plt.show()